# Hoff (8.2) Sensitivity Analysis


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import invgamma

# Ensure reproducibility and configure display options
rng = np.random.default_rng(20240606)
pd.set_option('display.float_format', lambda x: f"{x:0.3f}")
plt.style.use('seaborn-v0_8')

# Study data
n_A = 16
n_B = 16
ybar_A = 75.2
ybar_B = 77.5
s_A = 7.3
s_B = 8.1

# Prior hyperparameters
mu_0 = 75.0
gamma0_sq = 100.0
a0 = 1.0
b0 = 100.0
delta0_grid = np.array([-4, -2, 0, 2, 4], dtype=float)
tau0_sq_grid = np.array([10, 50, 100, 500], dtype=float)

# Output directory for figures
fig_dir = Path('figures')
fig_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
from typing import Tuple

def gibbs_sampler(num_samples: int,
                  burn_in: int,
                  delta0: float,
                  tau0_sq: float,
                  rng: np.random.Generator):
    'Run a Gibbs sampler for the hierarchical two-sample model.'
    total_iterations = num_samples + burn_in
    mu_samples = np.empty(num_samples)
    delta_samples = np.empty(num_samples)
    sigma2_samples = np.empty(num_samples)

    mu = 0.5 * (ybar_A + ybar_B)
    delta = 0.5 * (ybar_A - ybar_B)
    sigma2 = 0.5 * (s_A ** 2 + s_B ** 2)

    ss_A = (n_A - 1) * (s_A ** 2)
    ss_B = (n_B - 1) * (s_B ** 2)

    sample_index = 0
    for iteration in range(total_iterations):
        resid_A = ss_A + n_A * ((ybar_A - (mu + delta)) ** 2)
        resid_B = ss_B + n_B * ((ybar_B - (mu - delta)) ** 2)
        a_n = a0 + 0.5 * (n_A + n_B)
        b_n = b0 + 0.5 * (resid_A + resid_B)
        sigma2 = invgamma.rvs(a=a_n, scale=b_n, random_state=rng)

        var_mu = 1.0 / (1.0 / gamma0_sq + (n_A + n_B) / sigma2)
        mean_mu = var_mu * (mu_0 / gamma0_sq + (n_A * (ybar_A - delta) + n_B * (ybar_B + delta)) / sigma2)
        mu = rng.normal(loc=mean_mu, scale=np.sqrt(var_mu))

        var_delta = 1.0 / (1.0 / tau0_sq + (n_A + n_B) / sigma2)
        mean_delta = var_delta * (delta0 / tau0_sq + (n_A * (ybar_A - mu) - n_B * (ybar_B - mu)) / sigma2)
        delta = rng.normal(loc=mean_delta, scale=np.sqrt(var_delta))

        if iteration >= burn_in:
            mu_samples[sample_index] = mu
            delta_samples[sample_index] = delta
            sigma2_samples[sample_index] = sigma2
            sample_index += 1

    return mu_samples, delta_samples, sigma2_samples


def prior_correlation(tau0_sq: float) -> float:
    return (gamma0_sq - tau0_sq) / (gamma0_sq + tau0_sq)


In [ ]:
# Part (a): Posterior sampling across prior scenarios
num_samples = 20000
burn_in = 5000

data_records = []
posterior_prob_matrix = np.zeros((delta0_grid.size, tau0_sq_grid.size))
posterior_corr_matrix = np.zeros_like(posterior_prob_matrix)
posterior_delta_mean = np.zeros_like(posterior_prob_matrix)

for i, delta0 in enumerate(delta0_grid):
    for j, tau0_sq in enumerate(tau0_sq_grid):
        mu_draws, delta_draws, sigma2_draws = gibbs_sampler(
            num_samples=num_samples,
            burn_in=burn_in,
            delta0=delta0,
            tau0_sq=tau0_sq,
            rng=rng
        )
        theta_A = mu_draws + delta_draws
        theta_B = mu_draws - delta_draws
        prob_delta_lt_zero = np.mean(delta_draws < 0)
        ci_lower, ci_upper = np.quantile(delta_draws, [0.025, 0.975])
        posterior_corr = np.corrcoef(theta_A, theta_B)[0, 1]

        data_records.append({
            'delta0': delta0,
            'tau0_sq': tau0_sq,
            'Pr_delta_lt_0': prob_delta_lt_zero,
            'delta_ci_lower': ci_lower,
            'delta_ci_upper': ci_upper,
            'prior_corr_thetaA_thetaB': prior_correlation(tau0_sq),
            'posterior_corr_thetaA_thetaB': posterior_corr,
            'posterior_mu_mean': np.mean(mu_draws),
            'posterior_mu_sd': np.std(mu_draws, ddof=1),
            'posterior_delta_mean': np.mean(delta_draws),
            'posterior_delta_sd': np.std(delta_draws, ddof=1),
            'posterior_sigma2_mean': np.mean(sigma2_draws),
            'posterior_sigma2_sd': np.std(sigma2_draws, ddof=1)
        })

        posterior_prob_matrix[i, j] = prob_delta_lt_zero
        posterior_corr_matrix[i, j] = posterior_corr
        posterior_delta_mean[i, j] = np.mean(delta_draws)

results_df = pd.DataFrame(data_records)
results_df


In [ ]:
# Visual summaries for part (a)
heatmap_index = pd.Index(delta0_grid, name=r'$\delta_0$')
heatmap_columns = pd.Index(tau0_sq_grid, name=r'$\tau_0^2$')
prob_df = pd.DataFrame(posterior_prob_matrix, index=heatmap_index, columns=heatmap_columns)
corr_df = pd.DataFrame(posterior_corr_matrix, index=heatmap_index, columns=heatmap_columns)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(prob_df, annot=True, fmt='.2f', cmap='viridis', ax=axes[0])
axes[0].set_title('Posterior Pr(δ < 0)')

sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Posterior Corr(θ_A, θ_B)')

plt.tight_layout()
prob_figure_path = fig_dir / 'posterior_probability_and_correlation.png'
plt.savefig(prob_figure_path, dpi=300, bbox_inches='tight')
plt.show()
prob_figure_path


In [ ]:
# Part (b): Communicating sensitivity results
min_prob_idx = np.unravel_index(np.argmin(posterior_prob_matrix), posterior_prob_matrix.shape)
max_prob_idx = np.unravel_index(np.argmax(posterior_prob_matrix), posterior_prob_matrix.shape)
min_prob = posterior_prob_matrix[min_prob_idx]
max_prob = posterior_prob_matrix[max_prob_idx]
min_prior = (delta0_grid[min_prob_idx[0]], tau0_sq_grid[min_prob_idx[1]])
max_prior = (delta0_grid[max_prob_idx[0]], tau0_sq_grid[max_prob_idx[1]])

summary_message = (
    f"Evidence that θ_A < θ_B can be tailored to different prior opinions as follows:

"
    f"- Even under a skeptical prior centered at δ₀ = {min_prior[0]:.0f} with τ₀² = {min_prior[1]:.0f}, "
    f"the posterior probability that δ < 0 is {min_prob:.3f}.
"
    f"- Under an optimistic prior centered at δ₀ = {max_prior[0]:.0f} with τ₀² = {max_prior[1]:.0f}, "
    f"the probability that δ < 0 increases to {max_prob:.3f}.
"
    f"- Across all scenarios, the 95% credible intervals for δ remain "
    f"{results_df['delta_ci_lower'].min():.2f} to {results_df['delta_ci_upper'].max():.2f}, "
    f"indicating that the posterior concentrates near negative values.
"
    f"- Prior correlations between θ_A and θ_B range from {results_df['prior_corr_thetaA_thetaB'].min():.2f} "
    f"to {results_df['prior_corr_thetaA_thetaB'].max():.2f}, while the posterior correlations stay between "
    f"{results_df['posterior_corr_thetaA_thetaB'].min():.2f} and {results_df['posterior_corr_thetaA_thetaB'].max():.2f}, "
    f"showing how the data dominate shared belief in practice.

"
    f"Presenting the table, credible intervals, and heatmap enables stakeholders to choose the prior that matches "
    f"their beliefs and still see that P(θ_A < θ_B | Y) remains high in every case."
)
print(summary_message)
